# Anomaly detection

Anomaly detection is used in finding unusual events.

In [1]:
import numpy as np

### Define Model

In [2]:
class AnomalyDetection:
    def __init__(self, epsilon=0.001):
        self.epsilon = epsilon
        

    def fit(self, X):
        self.means = np.mean(X, axis=0)
        self.variances = np.var(X, axis=0)
        

    def get_gaussian_probabilities(self, X):
        X = np.array(X)
        
        probs = (1 / np.sqrt(2 * np.pi * self.variances)) * (np.exp(-(np.square(X - self.means)) / (2 * self.variances)))

        total_probs = np.prod(probs, axis=1)

        return total_probs

    
    def predict(self, X):
        probs = self.get_gaussian_probabilities(X)

        return (probs < self.epsilon).astype(int)

### Choosing the best epsilon using cross validation

In [3]:
def choose_best_epsilon(X, X_val, y_val):
    model = AnomalyDetection()
    model.fit(X)
    
    p_val = model.get_gaussian_probabilities(X_val)

    best_f1_score = 0
    best_epsilon = None
    
    epsilons = np.linspace(min(p_val), max(p_val), 1000)

    for epsilon in epsilons:
        y_pred = (p_val < epsilon).astype(int)

        tp = np.sum((y_val == 1) & (y_pred == 1)) # y_val is 1 and prediction is 1
        fp = np.sum((y_val == 0) & (y_pred == 1)) # y_val is 0 and prediction is 1
        fn = np.sum((y_val == 1) & (y_pred == 0)) # y_val is 1 and prediction is 0

        if tp == 0:
            continue

        # Precision = TP / TP + FP
        precision = tp / (tp + fp)

        # Recall = TP / TP + FN
        recall = tp / (tp + fn)
        
        # F1 score = (2 * Precision * Recall) / (Precision + Recall)
        f1_score = (2 * precision * recall) / (precision + recall)

        if f1_score > best_f1_score:
            best_epsilon = epsilon
            best_f1_score = f1_score

    return best_epsilon

### Sample dataset

In [4]:
# Example dataset (2 features)
X_train = np.array([
    [10, 10],
    [11, 11],
    [10, 12],
    [9, 11],
    [10, 9]
])

# Validation set
X_val = np.array([
    [10, 10],
    [50, 50],  # anomaly
    [11, 10]
])

y_val = np.array([0, 1, 0])  # 1 = anomaly

In [5]:
epsilon = choose_best_epsilon(X_train, X_val, y_val)

print(f"The best epsilon is {epsilon}")

The best epsilon is 0.0002077502836733311


### Fit model

In [6]:
model = AnomalyDetection(epsilon)

model.fit(X_train)

### Prediction

In [7]:
model.predict(X_val)

array([0, 1, 0])